<a href="https://colab.research.google.com/github/raheelam98/DataAnalysis/blob/main/EDA/EDA_Sessions_RM/Data_Cleaning_Inconsistent_Data_Sec_4_RM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this notebook, we're going to learn how to clean up inconsistent text entries.

Let's get started!

# Get our environment set up

The first thing we'll need to do is load in the libraries and dataset we'll be using.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Do some preliminary text pre-processing

We'll begin by taking a quick look at the first few rows of the data.

Just looking at this, I can see some problems due to inconsistent data entry: ' Germany', and 'germany', for example, or ' New Zealand' and 'New Zealand'.

The first thing I'm going to do is make everything lower case (I can change it back at the end if I like) and remove any white spaces at the beginning and end of cells. Inconsistencies in capitalizations and trailing white spaces are very common in text data and you can fix a good 80% of your text data entry inconsistencies by doing this.

Next we're going to tackle more difficult inconsistencies.

# Use fuzzy matching to correct inconsistent data entry

Alright, let's take another look at the 'Country' column and see if there's any more data cleaning we need to do.

It does look like there is another inconsistency: 'southkorea' and 'south korea' should be the same.

We're going to use the [fuzzywuzzy](https://github.com/seatgeek/fuzzywuzzy) package to help identify which strings are closest to each other. This dataset is small enough that we could probably could correct errors by hand, but that approach doesn't scale well. (Would you want to correct a thousand errors by hand? What about ten thousand? Automating things as early as possible is generally a good idea. Plus, it’s fun!)

> **Fuzzy matching:** The process of automatically finding text strings that are very similar to the target string. In general, a string is considered "closer" to another one the fewer characters you'd need to change if you were transforming one string into another. So "apple" and "snapple" are two changes away from each other (add "s" and "n") while "in" and "on" and one change away (rplace "i" with "o"). You won't always be able to rely on fuzzy matching 100%, but it will usually end up saving you at least a little time.

Fuzzywuzzy returns a ratio given two strings. The closer the ratio is to 100, the smaller the edit distance between the two strings. Here, we're going to get the ten strings from our list of cities that have the closest distance to "south korea".

We can see that two of the items in the cities are very close to "south korea": "south korea" and "southkorea". Let's replace all rows in our "Country" column that have a match of > 95 with "south korea".

To do this, I'm going to write a function. (It's a good idea to write a general purpose function you can reuse if you think you might have to do a specific task more than once or twice. This keeps you from having to copy and paste code too often, which saves time and can help prevent mistakes.)

Function Definition:

replace_matches_in_column(df, column, string_to_match, min_match=95): This function takes four parameters:
df: The dataframe where replacements will occur.
column: The column within the dataframe where you want to check for matches.
string_to_match: The string you want to find matches for.
min_match: The minimum similarity percentage to consider a string a match (default is 95%).
Extract Unique Strings:

strings = df[column].unique(): Retrieves all unique values from the specified column of the dataframe.
Find Close Matches:

matches = fuzzywuzzy.process.extract(string_to_match, strings, limit=5): Uses fuzzy string matching to find the top 5 strings from strings that are most similar to string_to_match.
Filter Matches by Similarity Threshold:

close_matches = [matches[0] for matches in matches if matches[1] >= min_match]: Filters out the matches to include only those with a similarity score greater than or equal to min_match. It creates a list of just these matching strings.
Identify Rows with Close Matches:

rows_with_matches = df[column].isin(close_matches): Creates a boolean series that is True for rows where the value in the specified column is in the close_matches list.
Replace Matched Rows:

df.loc[rows_with_matches, column] = string_to_match: Replaces the values in the matched rows’ specified column with string_to_match.
Completion Message:

print("All done!"): Prints a message indicating that the function has completed its execution.

Now that we have a function, we can put it to the test!